In [1]:
import torch
print("Card hinh cua ban la:", torch.cuda.get_device_name(0))
print("PyTorch da nhan GPU chua?", torch.cuda.is_available())

Card hinh cua ban la: NVIDIA GeForce RTX 4060 Laptop GPU
PyTorch da nhan GPU chua? True


In [2]:
import os
import shutil
from sklearn.model_selection import train_test_split

# 1. Khai báo đường dẫn (Đổi lại cho đúng với folder trên ổ D của bạn)
# Ví dụ mình đang giả sử folder dataset gốc tên là 'dataset_goc' nằm trong folder project
dataset_path = r'D:\Study_FPTU\DAP391m\plate detection\dataset' 
images_path = os.path.join(dataset_path, 'image')
labels_path = os.path.join(dataset_path, 'label')

# Thư mục đích sau khi chia xong
output_base_path = r'D:\Study_FPTU\DAP391m\plate detection\dataset_split'

# Tạo sẵn các thư mục đích (nếu chưa có)
for subset in ['train', 'val']:
    os.makedirs(os.path.join(output_base_path, subset, 'images'), exist_ok=True)
    os.makedirs(os.path.join(output_base_path, subset, 'labels'), exist_ok=True)

# 2. GHÉP ĐÔI: Lấy danh sách tên file (không kèm đuôi) của cả 2 bên
# Lưu ý: Sửa '.jpg' thành '.png' nếu ảnh của bạn là đuôi png nhé
image_names = set([os.path.splitext(f)[0] for f in os.listdir(images_path) if f.endswith('.jpg')])
label_names = set([os.path.splitext(f)[0] for f in os.listdir(labels_path) if f.endswith('.txt')])

# Dùng phép giao (intersection) của Set để tìm các tên file có ở CẢ 2 bên
valid_filenames = list(image_names.intersection(label_names))

print(f"Tổng số ảnh quét được: {len(image_names)}")
print(f"Tổng số nhãn quét được: {len(label_names)}")
print(f"--> Số cặp Ảnh-Nhãn hợp lệ ghép được: {len(valid_filenames)}")
print("-" * 30)

# Kiểm tra dữ liệu hợp lệ trước khi chia
total = len(valid_filenames)
if total == 0:
    raise RuntimeError(
        "Không tìm thấy cặp ảnh-nhãn hợp lệ. Kiểm tra lại đường dẫn dataset và phần mở rộng tệp (jpg/txt)."
    )

# 3. CHIA TRAIN/VAL: 80% train, 20% val trên tập hợp lệ
train_filenames, val_filenames = train_test_split(valid_filenames, test_size=0.2, random_state=42)

print(f"Số lượng đưa vào Train: {len(train_filenames)}")
print(f"Số lượng đưa vào Val: {len(val_filenames)}")

# Hàm copy file siêu tốc trên ổ cứng
def copy_files(filenames, subset_name):
    dest_image_dir = os.path.join(output_base_path, subset_name, 'images')
    dest_label_dir = os.path.join(output_base_path, subset_name, 'labels')
    
    for fname in filenames:
        shutil.copy(os.path.join(images_path, fname + '.jpg'), os.path.join(dest_image_dir, fname + '.jpg'))
        shutil.copy(os.path.join(labels_path, fname + '.txt'), os.path.join(dest_label_dir, fname + '.txt'))

# 4. Thực thi copy
print("\nĐang copy data vào tập Train...")
copy_files(train_filenames, 'train')

print("Đang copy data vào tập Val...")
copy_files(val_filenames, 'val')

print("\nHoàn tất! Dataset đã sạch sẽ và sẵn sàng để train.")

Tổng số ảnh quét được: 9397
Tổng số nhãn quét được: 9397
--> Số cặp Ảnh-Nhãn hợp lệ ghép được: 9397
------------------------------
Số lượng đưa vào Train: 7517
Số lượng đưa vào Val: 1880

Đang copy data vào tập Train...
Đang copy data vào tập Val...

Hoàn tất! Dataset đã sạch sẽ và sẵn sàng để train.


In [1]:
import os
import glob
import cv2
import numpy as np

# Thư mục chứa nhãn GỐC của bạn (Nhớ trỏ đúng và nên có bản backup)
labels_dir = r'D:\Study_FPTU\DAP391m\plate detection\dataset\label' 

txt_files = glob.glob(os.path.join(labels_dir, '*.txt'))

bbox_to_obb_count = 0
poly_to_obb_count = 0
obb_kept_count = 0

print(f"Bắt đầu mổ sẻ {len(txt_files)} file nhãn...")

for txt_path in txt_files:
    with open(txt_path, 'r') as file:
        lines = file.readlines()
    
    new_lines = []
    modified = False
    
    for line in lines:
        parts = line.strip().split()
        if not parts: continue
            
        cls = parts[0]
        
        # TRƯỜNG HỢP 1: Bounding Box Thẳng (5 số)
        if len(parts) == 5:
            x_c, y_c, w, h = map(float, parts[1:])
            # Ép thành 4 góc của hình chữ nhật đứng
            x1, y1 = x_c - w/2, y_c - h/2
            x2, y2 = x_c + w/2, y_c - h/2
            x3, y3 = x_c + w/2, y_c + h/2
            x4, y4 = x_c - w/2, y_c + h/2
            
            coords = [x1, y1, x2, y2, x3, y3, x4, y4]
            new_lines.append(f"{cls} " + " ".join([f"{max(0.0, min(1.0, c)):.6f}" for c in coords]) + "\n")
            modified = True
            bbox_to_obb_count += 1
            
        # TRƯỜNG HỢP 2: Đã là OBB chuẩn (9 số)
        elif len(parts) == 9:
            new_lines.append(line)
            obb_kept_count += 1

        # TRƯỜNG HỢP 3: Polygon (Nhiều hơn 9 số) -> Ép thành OBB
        elif len(parts) > 9:
            # Lấy tất cả tọa độ, gom thành cặp (x, y)
            coords = list(map(float, parts[1:]))
            points = np.array([[coords[i], coords[i+1]] for i in range(0, len(coords), 2)], dtype=np.float32)
            
            # Dùng OpenCV tìm hình chữ nhật xoay nhỏ nhất (Minimum Area Rectangle) bao quanh các điểm
            # Lưu ý: minAreaRect yêu cầu tọa độ pixel, nhưng YOLO dùng tọa độ chuẩn hóa [0,1].
            # Vì chỉ tính tỷ lệ, ta có thể nhân lên một số lớn (VD: 1000) rồi chia lại.
            points_scaled = points * 1000.0 
            rect = cv2.minAreaRect(points_scaled)
            box = cv2.boxPoints(rect)
            box_normalized = box / 1000.0 # Trả về lại khoảng [0, 1]
            
            # Gom lại thành mảng 1 chiều 8 phần tử
            obb_coords = box_normalized.flatten().tolist()
            
            new_lines.append(f"{cls} " + " ".join([f"{max(0.0, min(1.0, c)):.6f}" for c in obb_coords]) + "\n")
            modified = True
            poly_to_obb_count += 1
            
    # Lưu lại file nếu có thay đổi
    if modified:
        with open(txt_path, 'w') as file:
            file.writelines(new_lines)

print("-" * 30)
print(f"✅ Đã ép Bbox (5 số) thành OBB: {bbox_to_obb_count} nhãn")
print(f"✅ Đã ép Polygon (>9 số) thành OBB: {poly_to_obb_count} nhãn")
print(f"⏭️ Giữ nguyên OBB chuẩn (9 số): {obb_kept_count} nhãn")
print("Hoàn tất! Giờ mọi thứ đã là OBB chuẩn.")

Bắt đầu mổ sẻ 9397 file nhãn...
------------------------------
✅ Đã ép Bbox (5 số) thành OBB: 0 nhãn
✅ Đã ép Polygon (>9 số) thành OBB: 0 nhãn
⏭️ Giữ nguyên OBB chuẩn (9 số): 11324 nhãn
Hoàn tất! Giờ mọi thứ đã là OBB chuẩn.


In [3]:
from ultralytics import YOLO
import torch, os

print("GPU available?", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Using", torch.cuda.get_device_name(0))

yaml_path = r'D:\Study_FPTU\DAP391m\plate detection\dataset.yaml'
if not os.path.isfile(yaml_path):
    raise FileNotFoundError(f"data config not found: {yaml_path}")

model_file = 'yolov8n-obb.pt'   # or 'yolov8n.pt' if your labels are 4‑tupples
if not os.path.isfile(model_file):
    raise FileNotFoundError(f"model weights not found: {model_file}")

model = YOLO(model_file)

results = model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,        # change to 'cpu' or omit if no GPU
    workers=4,
    project='runs/train',
    name='plate_model_v1',
    patience=30
)

print("🎉 Quá trình train đã hoàn tất!")

GPU available? True
Using NVIDIA GeForce RTX 4060 Laptop GPU
Ultralytics 8.4.17  Python-3.10.0 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\Study_FPTU\DAP391m\plate detection\dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-obb.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=plate_mo

In [5]:
import torch
import paddle

print("--- KIỂM TRA YOLO (PyTorch) ---")
if torch.cuda.is_available():
    print(f"✅ YOLO đang dùng GPU: {torch.cuda.get_device_name(0)}")
else:
    print("❌ YOLO đang dùng CPU (Cần cài lại torch với CUDA)")

print("\n--- KIỂM TRA OCR (PaddlePaddle) ---")
# Kiểm tra xem Paddle có hỗ trợ GPU không
gpu_available = paddle.device.is_compiled_with_cuda()
if gpu_available:
    # Thử lấy danh sách thiết bị
    print(f"✅ PaddleOCR có hỗ trợ GPU!")
    print(f"Thiết bị hiện tại: {paddle.get_device()}")
else:
    print("❌ PaddleOCR đang dùng CPU")
    print("👉 Hãy chạy: pip install paddlepaddle-gpu -i https://mirror.baidu.com/pypi/simple")

print("\n--- KIỂM TRA THỰC TẾ TRÊN MODEL ---")
try:
    from ultralytics import YOLO
    model = YOLO("yolov8n.pt") # load tạm 1 model nhỏ
    print(f"YOLO Model Device: {model.device}")
except:
    print("Chưa cài Ultralytics")

--- KIỂM TRA YOLO (PyTorch) ---
✅ YOLO đang dùng GPU: NVIDIA GeForce RTX 4060 Laptop GPU

--- KIỂM TRA OCR (PaddlePaddle) ---
❌ PaddleOCR đang dùng CPU
👉 Hãy chạy: pip install paddlepaddle-gpu -i https://mirror.baidu.com/pypi/simple

--- KIỂM TRA THỰC TẾ TRÊN MODEL ---
YOLO Model Device: cpu
